## Model Monitoring

Tracks every model run logged to MLflow, compares each run's MAPE against the best earlier run of the same model, and flags drift when MAPE rises by more than a configurable threshold.

**Outputs**
- `retail_project.gold.model_metrics_history` - one row per MLflow run (never duplicated)
- `retail_project.gold.model_drift_status` - same runs plus baseline, % change and `drift_flag`
- Two CSV exports for the Grafana dashboard (`grafana/`)

**Flow:** sync runs -> compute drift -> simulate drift on noisy data -> re-sync -> export CSVs

In [0]:
%pip install xgboost

### Configuration
Change these values here only. `DRIFT_THRESHOLD` is **relative**: 0.15 means MAPE has to be 15% worse than the baseline (e.g. 10.41% -> 11.97%), not 15 percentage points worse.

In [0]:
EXPERIMENT_NAME = "/Shared/retail_demand_forecasting"
MAPE_METRIC = "mape"
RMSE_METRIC = "rmse"

DRIFT_THRESHOLD = 0.15          # flag when MAPE is more than 15% worse than the baseline

MODEL_URI = "models:/retail_project.gold.retail_demand_xgboost@champion"
SIM_NOISE_SIGMA = 0.5           # std-dev of the multiplicative noise used to simulate drift
SIM_NOISE_COLS = ["Sales_Lag1", "Sales_Lag7", "Sales_RollingAvg7"]

METRICS_TABLE = "retail_project.gold.model_metrics_history"
DRIFT_TABLE = "retail_project.gold.model_drift_status"
EXPORT_VOLUME = "retail_project.gold.monitoring_exports"
EXPORT_DIR = "/Volumes/retail_project/gold/monitoring_exports"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {METRICS_TABLE} (
    run_id STRING,
    run_name STRING,
    model_type STRING,
    run_timestamp TIMESTAMP,
    mape DOUBLE,
    rmse DOUBLE,
    is_simulated BOOLEAN,
    experiment_name STRING,
    ingested_at TIMESTAMP
) USING DELTA
""")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {EXPORT_VOLUME}")
print("Config loaded, tables and export volume ready.")

### Step 1: Sync MLflow runs into Delta
Pulls all finished runs from the experiment and inserts only the ones not already in `model_metrics_history` (`MERGE ... WHEN NOT MATCHED`), so re-running this cell never creates duplicates.

`model_type` is derived from the run name (`prophet_*` -> `prophet`, `xgboost_*` -> `xgboost`), because only the XGBoost run logged a `model_type` param.

In [0]:
import numpy as np
import pandas as pd
import mlflow
from datetime import datetime, timezone
from pyspark.sql import functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               TimestampType, BooleanType)

RUN_SCHEMA = StructType([
    StructField("run_id", StringType()),
    StructField("run_name", StringType()),
    StructField("model_type", StringType()),
    StructField("run_timestamp", TimestampType()),
    StructField("mape", DoubleType()),
    StructField("rmse", DoubleType()),
    StructField("is_simulated", BooleanType()),
    StructField("experiment_name", StringType()),
    StructField("ingested_at", TimestampType()),
])


def infer_model_type(run_name, param_type):
    text = f"{run_name} {param_type or ''}".lower()
    if "prophet" in text:
        return "prophet"
    if "xgb" in text:
        return "xgboost"
    return "unknown"


def fetch_runs():
    runs = mlflow.search_runs(
        experiment_names=[EXPERIMENT_NAME],
        filter_string="attributes.status = 'FINISHED'",
        output_format="pandas",
    )
    mape_col, rmse_col = f"metrics.{MAPE_METRIC}", f"metrics.{RMSE_METRIC}"
    missing = [c for c in (mape_col, rmse_col) if c not in runs.columns]
    if missing:
        available = [c for c in runs.columns if c.startswith("metrics.")]
        raise ValueError(f"Metric column(s) {missing} not found. Metrics in this experiment: {available}")
    return runs[runs[mape_col].notna()].copy(), mape_col, rmse_col


def sync_metrics_history():
    runs, mape_col, rmse_col = fetch_runs()
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    names = runs["tags.mlflow.runName"] if "tags.mlflow.runName" in runs.columns else pd.Series("", index=runs.index)
    ptypes = runs["params.model_type"] if "params.model_type" in runs.columns else pd.Series("", index=runs.index)
    sim_tag = runs["tags.simulated_drift"] if "tags.simulated_drift" in runs.columns else pd.Series("", index=runs.index)
    started = pd.to_datetime(runs["start_time"], utc=True).dt.tz_localize(None)

    records = []
    for i in runs.index:
        rmse_val = runs.at[i, rmse_col]
        records.append((
            str(runs.at[i, "run_id"]),
            str(names[i]) if pd.notna(names[i]) else "",
            infer_model_type(names[i], ptypes[i]),
            started[i].to_pydatetime(),
            float(runs.at[i, mape_col]),
            float(rmse_val) if pd.notna(rmse_val) else None,
            str(sim_tag[i]).lower() == "true",
            EXPERIMENT_NAME,
            now,
        ))

    before = spark.table(METRICS_TABLE).count()
    spark.createDataFrame(records, RUN_SCHEMA).createOrReplaceTempView("incoming_runs")
    spark.sql(f"""
        MERGE INTO {METRICS_TABLE} t
        USING incoming_runs s ON t.run_id = s.run_id
        WHEN NOT MATCHED THEN INSERT *
    """)
    after = spark.table(METRICS_TABLE).count()
    print(f"MLflow runs found: {len(records)} | newly inserted: {after - before} | table rows: {after}")


sync_metrics_history()
display(spark.table(METRICS_TABLE).orderBy("model_type", "run_timestamp"))

### Step 2: Drift status
For each run, the baseline is the **lowest MAPE among earlier runs of the same `model_type`** (the first run of a model is its own baseline). `drift_flag` is true when `(mape - baseline_mape) / baseline_mape > DRIFT_THRESHOLD`.

Note: the Prophet runs are for Store 1 only and XGBoost covers all stores, which is why each model type is compared only against itself.

In [0]:
def compute_drift_status(threshold=DRIFT_THRESHOLD):
    hist = spark.table(METRICS_TABLE)
    earlier = (Window.partitionBy("model_type")
               .orderBy("run_timestamp", "run_id")
               .rowsBetween(Window.unboundedPreceding, -1))

    out = (hist
           .withColumn("_best", F.min(F.struct("mape", "run_id")).over(earlier))
           .withColumn("baseline_mape", F.coalesce(F.col("_best.mape"), F.col("mape")))
           .withColumn("baseline_run_id", F.coalesce(F.col("_best.run_id"), F.col("run_id")))
           .withColumn("mape_change_pct", (F.col("mape") - F.col("baseline_mape")) / F.col("baseline_mape"))
           .withColumn("drift_threshold", F.lit(float(threshold)))
           .withColumn("drift_flag", F.col("mape_change_pct") > F.lit(float(threshold)))
           .withColumn("checked_at", F.current_timestamp())
           .select("run_id", "run_name", "model_type", "run_timestamp", "mape", "rmse",
                   "is_simulated", "baseline_run_id", "baseline_mape", "mape_change_pct",
                   "drift_threshold", "drift_flag", "checked_at"))

    (out.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(DRIFT_TABLE))
    return spark.table(DRIFT_TABLE)


drift_df = compute_drift_status()
display(drift_df.orderBy("model_type", "run_timestamp"))

### Step 3: Simulate drift
Loads the registered `champion` XGBoost model and evaluates it on the same held-out slice used in notebook 05 (last 15% of dates, open days only), first on clean data and then with multiplicative noise on the lag/rolling-average features. This mimics real-world drift, where the inputs the model sees no longer look like its training data. The noisy evaluation is logged as a new MLflow run tagged `simulated_drift=true`.

In [0]:
import mlflow.xgboost
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

FEATURE_COLS = [
    "Store", "DayOfWeek", "Open", "Promo", "SchoolHoliday", "IsStateHoliday",
    "Year", "Month", "WeekOfYear", "Sales_Lag1", "Sales_Lag7", "Sales_RollingAvg7",
    "TemperatureMean", "PrecipitationSum", "CompetitionDistance",
    "CompetitionOpenSinceMonth", "CompetitionOpenSinceYear",
    "Promo2", "Promo2SinceWeek", "Promo2SinceYear",
    "StoreType_Index", "Assortment_Index", "StateHoliday_Index",
]

gold = spark.table("retail_project.gold.sales_features").select("Date", "Sales", *FEATURE_COLS)
pdf = gold.toPandas().sort_values("Date", kind="stable").reset_index(drop=True)
test = pdf.iloc[int(len(pdf) * 0.85):]
test = test[test["Open"] == 1]
X_test, y_test = test[FEATURE_COLS], test["Sales"]
print("Evaluation rows:", len(test))

model = mlflow.xgboost.load_model(MODEL_URI)


def evaluate(model, X, y):
    preds = model.predict(X)
    mask = y > 0
    mape = float(mean_absolute_percentage_error(y[mask], preds[mask]))
    rmse = float(np.sqrt(mean_squared_error(y, preds)))
    return mape, rmse


clean_mape, clean_rmse = evaluate(model, X_test, y_test)
print(f"Clean data -> MAPE: {clean_mape:.2%}, RMSE: {clean_rmse:.2f}")

In [0]:
def add_noise(X, cols, sigma, seed=42):
    rng = np.random.default_rng(seed)
    Xn = X.copy()
    for c in cols:
        factor = 1 + rng.normal(0, sigma, size=len(Xn))
        noisy = np.clip(Xn[c].to_numpy() * factor, 0, None)
        if np.issubdtype(Xn[c].dtype, np.integer):
            noisy = noisy.round().astype(Xn[c].dtype)
        Xn[c] = noisy
    return Xn


X_noisy = add_noise(X_test, SIM_NOISE_COLS, SIM_NOISE_SIGMA)
noisy_mape, noisy_rmse = evaluate(model, X_noisy, y_test)
print(f"Noisy data (sigma={SIM_NOISE_SIGMA}) -> MAPE: {noisy_mape:.2%}, RMSE: {noisy_rmse:.2f}")

mlflow.set_experiment(EXPERIMENT_NAME)
with mlflow.start_run(run_name=f"xgboost_drift_sim_noise_{SIM_NOISE_SIGMA}"):
    mlflow.set_tag("simulated_drift", "true")
    mlflow.log_param("model_type", "XGBRegressor")
    mlflow.log_param("model_uri", MODEL_URI)
    mlflow.log_param("noise_sigma", SIM_NOISE_SIGMA)
    mlflow.log_param("noise_cols", ",".join(SIM_NOISE_COLS))
    mlflow.log_metric(MAPE_METRIC, noisy_mape)
    mlflow.log_metric(RMSE_METRIC, noisy_rmse)
    mlflow.log_metric("mape_clean_reference", clean_mape)

print("Simulated-drift run logged to MLflow.")

### Re-sync and check the drift flag
Picks up the new run, recomputes the baseline comparison, and shows the simulated run. If `drift_flag` is false, increase `SIM_NOISE_SIGMA` in the config cell and re-run Step 3 (each run adds a new MLflow run).

In [0]:
sync_metrics_history()
drift_df = compute_drift_status()

display(drift_df.orderBy("model_type", "run_timestamp"))

latest_sim = (drift_df.filter("is_simulated")
              .orderBy(F.col("run_timestamp").desc()).limit(1).collect())
if latest_sim:
    r = latest_sim[0]
    print(f"Latest simulated run: MAPE {r['mape']:.2%} vs baseline {r['baseline_mape']:.2%} "
          f"({r['mape_change_pct']:+.1%}), threshold {r['drift_threshold']:.0%} -> drift_flag = {r['drift_flag']}")
    if not r["drift_flag"]:
        print("Flag did not trigger: raise SIM_NOISE_SIGMA and re-run Step 3.")

### Step 4: Export to CSV
Writes both tables as CSV files into the Unity Catalog volume `retail_project.gold.monitoring_exports`. Download them from **Catalog -> retail_project -> gold -> Volumes -> monitoring_exports** and place them in `grafana/data/` in the repo.

In [0]:
def export_csv(table_name, filename):
    pdf_out = spark.table(table_name).orderBy("model_type", "run_timestamp").toPandas()
    path = f"{EXPORT_DIR}/{filename}"
    pdf_out.to_csv(path, index=False)
    print(f"{table_name}: {len(pdf_out)} rows -> {path}")


export_csv(METRICS_TABLE, "model_metrics_history.csv")
export_csv(DRIFT_TABLE, "model_drift_status.csv")